# Probability Foundations Part 1

## Probability Models, Random Variables, and Distributions

This notebook summarizes the first part of the Probability Foundations learning session.

It is not a raw transcript. It is a structured learning artifact combining:

- definitions;
- conceptual explanations;
- important equations;
- corrections and clarifications;
- small numerical examples;
- short Python simulations.

The main path is:

$$
\text{probability model}
\rightarrow
\text{random variable}
\rightarrow
\text{distribution}
\rightarrow
\text{joint distribution}
\rightarrow
\text{conditioning and independence}
\rightarrow
\text{transformation and simulation}
$$


## 1. Probability Models

A probability model can be represented as:

$$
(S,\mathcal{F},P)
$$

where:

- $S$ is the sample space;
- $\mathcal{F}$ is the collection of events;
- $P$ is a probability measure.

The key conceptual upgrade in this session was to view probability as a **measure**.

A probability measure assigns probability mass to events:

$$
P:\mathcal{F}\to[0,1].
$$

It satisfies:

$$
P(S)=1
$$

and for disjoint events $A_1,A_2,\dots$,

$$
P\left(\bigcup_i A_i\right)=\sum_i P(A_i).
$$

This explains why probability behaves like length, area, or volume, except that the whole sample space has total mass 1.


## 2. Inclusion-Exclusion

For two events:

$$
P(A\cup B)=P(A)+P(B)-P(A\cap B).
$$

For $n$ events:

$$
P\left(\bigcup_{i=1}^n A_i\right)
=
\sum_{k=1}^n(-1)^{k+1}
\sum_{1\le i_1<\cdots<i_k\le n}
P(A_{i_1}\cap\cdots\cap A_{i_k}).
$$

The alternating signs correct overcounting.

If an outcome belongs to exactly $r$ events, then it is counted:

$$
\binom r1-\binom r2+\binom r3-\cdots+(-1)^{r+1}\binom rr=1.
$$


In [ ]:
from math import comb

def inclusion_exclusion_count(r):
    return sum((-1)**(k+1) * comb(r, k) for k in range(1, r+1))

for r in range(1, 8):
    print(r, inclusion_exclusion_count(r))


## 3. Birthday Problem

For $C$ people and 365 equally likely birthdays, the probability that at least one pair shares a birthday is:

$$
1-\frac{365!}{(365-C)!\,365^C}.
$$

The smallest $C$ for which this exceeds $0.5$ is $23$.

The intuitive reason is that the number of possible pairs grows quickly:

$$
\binom{C}{2}.
$$

For $C=23$:

$$
\binom{23}{2}=253.
$$


In [ ]:
import math

def birthday_match_probability(C, days=365):
    if C > days:
        return 1.0
    no_match = 1.0
    for k in range(C):
        no_match *= (days - k) / days
    return 1 - no_match

for C in [2, 10, 20, 22, 23, 30, 50]:
    print(C, birthday_match_probability(C))


## 4. Conditional Probability and Bayes' Theorem

Conditional probability is:

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
$$

for $P(B)>0$.

Bayes' theorem is:

$$
P(A\mid B)=\frac{P(A)}{P(B)}P(B\mid A).
$$

The important conceptual interpretation from the session was:

> Bayes' theorem translates between the viewpoint conditioned on $A$ and the viewpoint conditioned on $B$, using the same joint event $A\cap B$ but normalizing by different base probabilities.


## 5. Independence

Two events $A$ and $B$ are independent if:

$$
P(A\cap B)=P(A)P(B).
$$

Equivalently, if $P(B)>0$,

$$
P(A\mid B)=P(A).
$$

So independence means that knowing $B$ occurred does not change the probability assigned to $A$.

In real-world modeling, independence is usually not known absolutely. It is justified by assumptions, mechanism, experimental design, or empirical evidence.


## 6. Random Variables

A random variable is a function:

$$
X:S\to\mathbb{R}.
$$

It numerically encodes outcomes.

For example:

$$
S=\{\text{rain},\text{snow},\text{clear}\}
$$

and

$$
X(\text{rain})=3,\quad X(\text{snow})=6,\quad X(\text{clear})=-2.7.
$$

Then the event

$$
\{X<5\}
$$

means:

$$
\{s\in S:X(s)<5\}=\{\text{rain},\text{clear}\}.
$$

Important clarification:

> The function $X$ is fixed. The randomness comes from the unknown outcome $s$.


In [ ]:
S = ["rain", "snow", "clear"]
X = {"rain": 3, "snow": 6, "clear": -2.7}

event_X_less_than_5 = [s for s in S if X[s] < 5]
event_X_less_than_5


## 7. Discrete Distributions

For a discrete random variable, the probability mass function is:

$$
p_X(x)=P(X=x).
$$

Important examples include Bernoulli, binomial, geometric, Poisson, negative binomial, hypergeometric, and multinomial distributions.

For a binomial random variable:

$$
X\sim\operatorname{Binomial}(n,p),
$$

the probability mass function is:

$$
P(X=x)=\binom{n}{x}p^x(1-p)^{n-x}.
$$

Its mean and variance are:

$$
\mathbb{E}[X]=np
$$

$$
\operatorname{Var}(X)=np(1-p).
$$


In [ ]:
import numpy as np
import pandas as pd
from math import comb

def binomial_pmf(n, p):
    return pd.DataFrame({
        "x": list(range(n + 1)),
        "P(X=x)": [comb(n, x) * (p ** x) * ((1 - p) ** (n - x)) for x in range(n + 1)]
    })

binomial_pmf(20, 0.2).head(10)


### Binomial Concentration Insight

For fixed $n$, the variance

$$
np(1-p)
$$

is largest at $p=1/2$ and smaller when $p$ is closer to $0$ or $1$.

So a more biased coin produces a count distribution that is more concentrated around its expected count.

Because the probabilities must sum to 1, greater concentration often produces a higher peak probability.


In [ ]:
for p in [0.2, 0.5, 0.8]:
    pmf = binomial_pmf(20, p)
    max_row = pmf.loc[pmf["P(X=x)"].idxmax()]
    print(f"p={p}: mode={int(max_row['x'])}, peak probability={max_row['P(X=x)']:.4f}, variance={20*p*(1-p):.2f}")


## 8. Continuous Distributions

A continuous random variable satisfies:

$$
P(X=x)=0
$$

for every exact point $x$.

An absolutely continuous random variable has a density $f_X$ such that:

$$
P(a\le X\le b)=\int_a^b f_X(x)\,dx.
$$

A density satisfies:

$$
f_X(x)\ge0
$$

and

$$
\int_{-\infty}^{\infty}f_X(x)\,dx=1.
$$

Important distinction:

> Density is not probability. Probability is area under the density curve.


## 9. Cumulative Distribution Functions

The CDF of $X$ is:

$$
F_X(x)=P(X\le x).
$$

It works for discrete, continuous, and mixed distributions.

For discrete variables:

$$
F_X(x)=\sum_{y\le x}P(X=y).
$$

For absolutely continuous variables:

$$
F_X(x)=\int_{-\infty}^x f_X(t)\,dt.
$$

Where differentiable:

$$
f_X(x)=F_X'(x).
$$

The CDF is more general than a PMF or density because it can represent all distribution types.


## 10. Change of Variable

If

$$
Y=h(X),
$$

then the distribution of $Y$ is determined by the distribution of $X$ and the transformation $h$.

For discrete $X$:

$$
P(Y=y)=\sum_{x:h(x)=y}P(X=x).
$$

For continuous $X$ and strictly monotone differentiable $h$:

$$
f_Y(y)=
\frac{f_X(h^{-1}(y))}
{|h'(h^{-1}(y))|}.
$$

The derivative corrects for local stretching or compression.


## 11. Joint Distributions

The marginal distributions of $X$ and $Y$ do not determine their relationship.

The joint distribution stores probabilities:

$$
P((X,Y)\in B),
\qquad B\subseteq\mathbb{R}^2.
$$

The joint CDF is:

$$
F_{X,Y}(x,y)=P(X\le x,\;Y\le y).
$$

For discrete variables:

$$
p_{X,Y}(x,y)=P(X=x,\;Y=y).
$$

For continuous variables:

$$
P(a\le X\le b,\;c\le Y\le d)
=
\int_c^d\int_a^b f_{X,Y}(x,y)\,dx\,dy.
$$

Marginals can be obtained by summing or integrating out the other variable.


## 12. Conditional Distributions and Independence

For discrete variables:

$$
p_{Y\mid X}(y\mid x)=\frac{p_{X,Y}(x,y)}{p_X(x)}.
$$

For continuous variables:

$$
f_{Y\mid X}(y\mid x)=\frac{f_{X,Y}(x,y)}{f_X(x)}.
$$

Independence means:

$$
P(X\in B_1,\;Y\in B_2)=P(X\in B_1)P(Y\in B_2).
$$

For discrete variables:

$$
p_{X,Y}(x,y)=p_X(x)p_Y(y).
$$

For continuous variables:

$$
f_{X,Y}(x,y)=f_X(x)f_Y(y).
$$

Equivalent interpretation:

> $X$ and $Y$ are independent if conditioning on one does not change the distribution of the other.


## 13. I.I.D. Samples

A sequence

$$
X_1,\dots,X_n
$$

is i.i.d. if the random variables are independent and each has the same distribution.

For continuous variables with common density $f$:

$$
f_{X_1,\dots,X_n}(x_1,\dots,x_n)=f(x_1)\cdots f(x_n).
$$

This product structure is one of the main reasons i.i.d. samples are central in statistics and likelihood inference.


## 14. Multidimensional Change of Variable and Jacobian

For transformations

$$
Z=h_1(X,Y),\qquad W=h_2(X,Y),
$$

the continuous change-of-variable formula uses the Jacobian determinant:

$$
J(x,y)=
\det
\begin{pmatrix}
\frac{\partial h_1}{\partial x} & \frac{\partial h_1}{\partial y}\\
\frac{\partial h_2}{\partial x} & \frac{\partial h_2}{\partial y}
\end{pmatrix}.
$$

The transformed density is:

$$
f_{Z,W}(z,w)
=
\frac{
f_{X,Y}(h^{-1}(z,w))
}{
|J(h^{-1}(z,w))|
}.
$$

Interpretation:

> The Jacobian determinant is the local area-scaling factor of a multidimensional transformation.


In [ ]:
# Example: h(x, y) = (2x, 3y)
# The Jacobian matrix is [[2, 0], [0, 3]], so the determinant is 6.

J = np.linalg.det(np.array([[2, 0], [0, 3]]))
J


## 15. Convolution

If

$$
Z=X+Y
$$

and $X,Y$ are independent, then the distribution of $Z$ is the convolution of the distributions of $X$ and $Y$.

Discrete convolution:

$$
p_Z(z)=\sum_w p_X(z-w)p_Y(w).
$$

Continuous convolution:

$$
f_Z(z)=\int_{-\infty}^{\infty}f_X(z-w)f_Y(w)\,dw.
$$

Key insight:

> Convolution aggregates all compatible ways two quantities can combine to a target value.

In probability, $w$ and $z-w$ are meaningful because:

$$
z=(z-w)+w.
$$


In [ ]:
# Discrete convolution example:
# X ~ Binomial(4, 1/5), Y ~ Bernoulli(1/4), independent.
# Compute P(Z=3), where Z = X + Y.

p = 1/5
theta = 1/4

def binom_prob(n, p, x):
    return comb(n, x) * (p ** x) * ((1 - p) ** (n - x))

P_Z_3 = binom_prob(4, p, 3) * (1 - theta) + binom_prob(4, p, 2) * theta
P_Z_3


## 16. Simulation

Most computational simulation starts from pseudorandom values treated as:

$$
U_1,U_2,\dots\sim\operatorname{i.i.d.}\operatorname{Uniform}[0,1].
$$

A uniform variable can be transformed into other distributions.

For $X\sim\operatorname{Uniform}[L,R]$:

$$
X=(R-L)U+L.
$$

For $X\sim\operatorname{Bernoulli}(\theta)$:

$$
X=
\begin{cases}
1,&U\le\theta,\\
0,&U>\theta.
\end{cases}
$$

For inverse-CDF sampling:

$$
Y=F^{-1}(U)
$$

has CDF $F$.


In [ ]:
rng = np.random.default_rng(42)
N = 100_000

U = rng.uniform(0, 1, N)

# Simulate Bernoulli(1/3)
bernoulli = (U <= 1/3).astype(int)

bernoulli.mean(), bernoulli.var()


## 17. Computer Exercise 2.10.10

The exercise asks us to simulate several distributions and compute:

$$
\bar X=\frac{1}{N}\sum_{i=1}^N X_i
$$

and

$$
\frac{1}{N}\sum_{i=1}^N (X_i-\bar X)^2.
$$

We use $N=100{,}000$.

Note: the textbook's geometric distribution counts failures before the first success. NumPy's `geometric` counts trials until first success, so we subtract 1.


In [ ]:
rng = np.random.default_rng(42)
N = 100_000

samples = {
    "Uniform[0, 1]": rng.uniform(0, 1, N),
    "Uniform[5, 8]": rng.uniform(5, 8, N),
    "Bernoulli(1/3)": rng.binomial(1, 1/3, N),
    "Binomial(12, 1/3)": rng.binomial(12, 1/3, N),
    "Geometric(1/5)": rng.geometric(1/5, N) - 1,
    "Exponential(1)": rng.exponential(scale=1, size=N),
    "Exponential(13)": rng.exponential(scale=1/13, size=N),
    "N(0, 1)": rng.normal(0, 1, N),
    "N(5, 9)": rng.normal(5, 3, N),
}

theory = {
    "Uniform[0, 1]": (1/2, 1/12),
    "Uniform[5, 8]": (6.5, 9/12),
    "Bernoulli(1/3)": (1/3, (1/3)*(2/3)),
    "Binomial(12, 1/3)": (4, 12*(1/3)*(2/3)),
    "Geometric(1/5)": (4, 20),
    "Exponential(1)": (1, 1),
    "Exponential(13)": (1/13, 1/(13**2)),
    "N(0, 1)": (0, 1),
    "N(5, 9)": (5, 9),
}

rows = []

for name, x in samples.items():
    empirical_mean = np.mean(x)
    empirical_variance = np.mean((x - empirical_mean) ** 2)
    theoretical_mean, theoretical_variance = theory[name]

    rows.append({
        "Distribution": name,
        "Empirical mean": empirical_mean,
        "Theoretical mean": theoretical_mean,
        "Mean error": empirical_mean - theoretical_mean,
        "Empirical variance": empirical_variance,
        "Theoretical variance": theoretical_variance,
        "Variance error": empirical_variance - theoretical_variance,
    })

results = pd.DataFrame(rows)
results


## 18. Misconceptions and Corrections

Important corrections from this session:

1. Probability is a measure on events, not only a number attached to uncertainty.
2. Uniform probability is only one possible probability measure.
3. Bayes' theorem relates $P(A\mid B)$ and $P(B\mid A)$, but they are not generally equal.
4. Exact point probabilities are zero for continuous random variables, so densities must be integrated to get probabilities.
5. Lower variance can explain higher peak probability inside the binomial family, but this is not a universal theorem for all distributions.
6. The geometric distribution has multiple conventions; always check whether it counts failures or trials.
7. For Jupyter compatibility in this project, display equations should use `$$...$$`.


## 19. Forward Connections

This part prepares for later probability, statistics, and deep learning topics.

Important forward links:

- Expectation and variance build directly on distributions.
- I.i.d. samples lead naturally to likelihood functions.
- Conditional distributions support Bayesian inference and probabilistic modeling.
- Convolution connects to sums of random variables and later to convolutional operations.
- Change of variables connects to reparameterization tricks and normalizing flows.
- Simulation connects to Monte Carlo methods and approximate inference.


## 20. Next Part

Recommended next session:

# Probability Foundations Part 2: Expectation

Likely topics:

- expectation of discrete and continuous random variables;
- linearity of expectation;
- variance and standard deviation;
- covariance and correlation;
- conditional expectation;
- connections to empirical averages and learning algorithms.
